# OncoSeg — Verify the `fix/review-findings` branch on Colab

Runs the **post-fix** test + verification suite on a Colab **GPU** runtime. No dataset or checkpoint needed
for sections 1–6.

**Before running:** `Runtime → Change runtime type → Hardware accelerator → GPU`, then **Runtime → Run all**.

> **Why a one-time kernel restart?** Section 1 installs `monai`/`numpy` etc., but Colab keeps the *old* numpy
> loaded in the running kernel, which breaks in-kernel imports later (`cannot import name _center`). So section 1
> **restarts the kernel once** right after installing. When it does, just **Run all again** — it's idempotent
> (the clone is skipped, pip is a no-op, and it won't restart a second time). The test/smoke cells run as fresh
> subprocesses, so they always pick up the freshly-installed packages.


## 1 · Clone + install + (one-time) kernel restart

Installs `.[dev,serve,dicom]` — `monai[all]`, `nibabel`, `fastapi`, `python-multipart`, `pydicom`/`highdicom`,
`pytest`, `ruff` (the same extras CI uses, plus `dicom`). Takes ~2–3 min the first time.


In [ ]:
import os
REPO = "https://github.com/danielchen26/OncoSeg-3D-Multi-Scale-Tumor-Segmentation-for-Automated-Treatment-Response-Assessment.git"
BRANCH = "fix/review-findings"
FLAG = "/content/.oncoseg_installed"   # a FILE flag survives a kernel restart (env vars do NOT)
if not os.path.isdir("/content/oncoseg"):
    !git clone --branch $BRANCH --depth 1 $REPO /content/oncoseg
%cd /content/oncoseg
!git log --oneline -1
if not os.path.exists(FLAG):
    !pip -q install -e "/content/oncoseg[dev,serve,dicom]"
    open(FLAG, "w").close()
    print("\n>>> Installed. Restarting the kernel ONCE so fresh numpy/monai load — then run Runtime ▸ Run all again. <<<")
    import time; time.sleep(1)
    os.kill(os.getpid(), 9)   # hard-restart the Colab kernel
else:
    print("Dependencies already installed and kernel restarted — continuing.")

## 2 · Runtime + import check (after restart)


In [ ]:
import os; os.chdir('/content/oncoseg') if os.path.isdir('/content/oncoseg') else None
import sys, platform
print('Python:', sys.version.split()[0], '|', platform.platform())
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('--- optional deps ---')
for m in ['monai','nibabel','fastapi','pydicom','highdicom','scipy','numpy']:
    try: __import__(m); print(f'  {m}: OK')
    except Exception as e: print(f'  {m}: MISSING ({e})')

## 3 · Full test suite

Runs as a subprocess (`!pytest`), so it always uses the freshly-installed packages. With the `dev,serve,dicom`
extras present, the tests that *skip* on a bare machine (needing monai/nibabel/pydicom/highdicom) now **run for
real**. Expectation: **all pass, 0 failed, 0 errors** (≈194 passed).


In [ ]:
!cd /content/oncoseg && pytest tests/ -q -rs --tb=short

## 4 · Lint (ruff) — same check CI runs


In [ ]:
!cd /content/oncoseg && ruff check src/ tests/ && echo 'ruff: clean'

## 5 · Smoke-test the fixed algorithm code paths (real tensors, no dataset)

Runs as a **subprocess** (fresh interpreter — immune to the stale-kernel issue). Exercises, on random
weights + synthetic volumes, that each fixed path *runs* (not that it's accurate — accuracy needs training):

- **F04** MC-Dropout on the trained *inline* `train_all.OncoSeg` (used to crash on `self.model.decoder`).
- **F16** uncertainty is per-channel **binary** entropy, bounded by `ln 2 ≈ 0.693`.
- **F06** RECIST longest diameter scans **all** slices (phantom: longest extent on a non-max-area slice).
- **F09** `DeepSupervisionLoss` interpolates multi-scale predictions instead of crashing.
- **F08** best-checkpoint selection is **NaN-safe**.


In [ ]:
smoke = r'''import torch, numpy as np, math, sys
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| numpy", np.__version__, "| torch", torch.__version__)
ok = True

# F04 + F16: MC-Dropout on the INLINE train_all architecture (the trained one)
from train_all import OncoSeg as InlineOncoSeg
from src.inference import Predictor
model = InlineOncoSeg(in_channels=4, num_classes=3, embed_dim=24, depths=(2,2,2,2),
                      num_heads=(3,6,12,24), deep_supervision=False).to(device).eval()
assert hasattr(model, "decoders") and not hasattr(model, "decoder")
pred = Predictor(model=model, device=torch.device(device), roi_size=(64,64,64), mc_samples=4)
unc = pred._estimate_uncertainty(torch.rand(1,4,64,64,64, device=device))  # F04: must not raise
f16 = unc.max() <= math.log(2) + 1e-3
print(f"F04 MC-dropout ran (shape {unc.shape}) -> OK")
print(f"F16 entropy<=ln2? max={float(unc.max()):.4f} (ln2={math.log(2):.4f}) -> {'OK' if f16 else 'FAIL'}"); ok &= f16

# F06: RECIST longest diameter across all slices
from src.response.recist import RECISTMeasurer
m = RECISTMeasurer()
mask = np.zeros((64,64,8), np.uint8); mask[10:40,10:40,0]=1; mask[30,5:55,1]=1
d = m.longest_axial_diameter(mask, pixdim=(1.0,1.0,1.0))
f06 = d > 45
print(f"F06 longest diameter={d:.1f}mm (expect ~49, pre-fix ~41) -> {'OK' if f06 else 'FAIL'}"); ok &= f06

# F09: deep-supervision loss interpolates multi-scale predictions
from src.training.losses import DeepSupervisionLoss, DiceCELoss
ds = DeepSupervisionLoss(DiceCELoss())
tgt = torch.zeros(1,3,32,32,32, device=device); tgt[:,0]=1
preds = [torch.randn(1,3,32,32,32, device=device), torch.randn(1,3,16,16,16, device=device), torch.randn(1,3,8,8,8, device=device)]
loss = ds(preds, tgt)  # F09: must not raise a shape error
f09 = bool(torch.isfinite(loss)) and loss.dim()==0
print(f"F09 deep-supervision loss={float(loss):.4f} finite scalar -> {'OK' if f09 else 'FAIL'}"); ok &= f09

# F08: NaN-safe best-checkpoint selection
guarded = lambda metric, best: (not math.isnan(metric)) and metric > best
row = np.array([0.71, 0.67, np.nan])  # empty-ET subject -> NaN region
f08 = math.isnan(float(np.mean(row))) and guarded(float(np.nanmean(row)), 0.0)
print(f"F08 plain-mean NaN, nanmean={float(np.nanmean(row)):.4f}, saves with guard -> {'OK' if f08 else 'FAIL'}"); ok &= f08

print("\nSMOKE_RESULT:", "ALL OK" if ok else "SOME FAILED")
sys.exit(0 if ok else 1)
'''
with open('/content/_smoke.py','w') as f: f.write(smoke)
!cd /content/oncoseg && python /content/_smoke.py

## 6 · Re-derive the two CRITICAL statistics from the committed arrays

No model needed — recomputes the honest numbers the docs now report (F01 Wilcoxon, F02 foreground ECE,
F07 dominant failure region) directly from the committed `.npy` / JSON. Also a subprocess.


In [ ]:
stats = r'''import numpy as np, json
from scipy.stats import wilcoxon
o = np.load("experiments/local_results/oncoseg_per_subject_dice.npy")  # cols [TC, WT, ET]
u = np.load("experiments/local_results/unet3d_per_subject_dice.npy")
print("per-subject arrays:", o.shape, "(val n =", o.shape[0], "-> split 388/96)")
for i,name in enumerate(["TC","WT","ET"]):
    a,b = o[:,i], u[:,i]; mk = ~(np.isnan(a)|np.isnan(b)); a,b = a[mk], b[mk]
    p = wilcoxon(a, b, alternative="greater").pvalue
    print(f"  {name}: delta={(a-b).mean():+.4f}  p(OncoSeg>UNet3D)={p:.4f}  OncoSeg wins {int((a>b).sum())}/{int(mk.sum())}")
om, um = np.nanmean(o,axis=1), np.nanmean(u,axis=1); mm = ~(np.isnan(om)|np.isnan(um))
print("  MEAN p =", round(float(wilcoxon(om[mm],um[mm],alternative="greater").pvalue),4), "-> F01: NO region significant; WT favors UNet3D")
means = np.nanmean(o, axis=1); bottom = np.argsort(means)[:5]
opr, bpr = np.nanmean(o,axis=0), np.nanmean(o[bottom],axis=0)
rel = {n:(opr[i]-bpr[i])/opr[i] for i,n in enumerate(["TC","WT","ET"])}
print("  F07 relative drop (bottom-5):", {k:round(v,3) for k,v in rel.items()}, "-> dominant =", max(rel, key=rel.get))
d = json.load(open("experiments/local_results/uncertainty_metrics.json"))
print("  F02 pooled ECE =", d["ece_median_case"], "| foreground ECE =", d.get("ece_median_case_foreground"), "-> over-confident on tumor")
'''
with open('/content/_stats.py','w') as f: f.write(stats)
!cd /content/oncoseg && python /content/_stats.py

## 7 · (Optional, slow) Train end-to-end for a couple of epochs

Uncomment to exercise the **training loop** on the real MSD Brain Tumour dataset. Downloads **~7 GB** and
trains a few epochs on the GPU (tens of minutes). Verifies the seeded, NaN-guarded, correctly-labelled
training path runs end-to-end; it does **not** reproduce the paper's 50-epoch numbers.


In [ ]:
# # WARNING: downloads ~7GB and trains. Uncomment to run.
# !cd /content/oncoseg && python train_local.py --epochs 2 --val-interval 1 --seed 42

---
**How to read it:** §3 should report all tests passing (incl. the monai/nibabel/pydicom/highdicom ones that
skip locally), §4 `ruff: clean`, §5 prints `SMOKE_RESULT: ALL OK`, §6 reproduces the honest statistics.
If §3/§5 look off, paste their output back.
